# Tutorial rápido SPARQL para Consultas en Wikidata

python version: 3.12.7

## 1. Estructura Básica de una Consulta

```sparql
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?variable1 ?variable2
WHERE {
    # Patrones de búsqueda
}
LIMIT 50
```

### Componentes esenciales:
- **PREFIX**: Define abreviaturas para URIs largos
- **SELECT**: Especifica qué variables queremos en los resultados
- **WHERE**: Contiene los patrones de búsqueda (tripletas)
- **LIMIT**: Limita el número de resultados

## 2. Tripletas Básicas

Las tripletas siguen el patrón: `sujeto predicado objeto`

```sparql
?pelicula wdt:P31 wd:Q11424 .     # ?pelicula es una instancia de película
?pelicula wdt:P57 wd:Q25191 .     # ?pelicula tiene como director a Nolan
?pelicula rdfs:label ?titulo .    # obtiene el título de la película
```

Las propiedades (predicado) tienen el prefijo `wdt`. Las entidades el prefijo `wd`.

Puedes buscar los códigos en la página de wikidata: https://www.wikidata.org/

## 3. Obtener Etiquetas Legibles

Las entidades son objetos con distintas propiedades. Para obtener la etiqueta legible usa `rdfs:label`.

```sparql
?entidad rdfs:label ?nombre .
FILTER (lang(?nombre) = 'es')     # Solo en español
FILTER (lang(?nombre) = 'en')     # Solo en inglés
```

## 4. OPTIONAL - Datos Opcionales

Cuando un dato puede no existir para todas las entidades:

```sparql
OPTIONAL { ?pelicula wdt:P2047 ?duracion }
OPTIONAL { ?pelicula wdt:P2130 ?presupuesto }
```

Es muy útil porque normalmente faltan datos y si no se utiliza obtenemos consultas vacías.

## 5. BIND - Crear Nuevas Variables

```sparql
?pelicula wdt:P577 ?fecha .
BIND(YEAR(?fecha) AS ?año)        # Extrae solo el año de una fecha
```

## 6. FILTER - Filtrar Resultados

### Filtros de idioma:
```sparql
FILTER (lang(?titulo) = 'en')
```

Si no se utiliza se repetirán filas con el mismo resultado pero con la etiqueta correspondiente en todos los idiomas disponibles.

### Filtros numéricos:
```sparql
FILTER (?año >= 2015)
FILTER (?altura > 50)
FILTER (?altura > 1.70 && ?altura < 2.00)
FILTER (?año < 1700)
```

### Verificar si una variable tiene valor:
```sparql
FILTER (BOUND(?duracion))
FILTER (BOUND(?movimiento))
```
Obligamos a que no haya resultados sin ese valor.

## 7. Navegación por Jerarquías

Para incluir subclases en la búsqueda:

```sparql
?libro wdt:P31/wdt:P279* wd:Q7725634    # instancia de o subclase de obra literaria
?museo wdt:P31/wdt:P279* wd:Q33506       # instancia de o subclase de museo
```

Si solo se utiliza `wdt:P31` (instance of) solo se devuelven las instancias directas de la clase.

## 8. DISTINCT - Eliminar Duplicados

```sparql
SELECT DISTINCT ?pelicula ?titulo ?año
```

## 9. GROUP_CONCAT - Concatenar Múltiples Valores

Cuando una entidad tiene varios valores para una propiedad:

```sparql
SELECT ?titulo (GROUP_CONCAT(DISTINCT ?ingrediente; separator=", ") AS ?ingredientes)
WHERE {
    # consulta para ?ingrediente
}
GROUP BY ?titulo
```

Así evitamos que se dupliquen filas con los valores de esa propiedad.

## 10. Agregaciones

### Contar elementos:
```sparql
SELECT ?equipo (COUNT(?titulo) AS ?titulos){
    # consulta para titulo
}
GROUP BY ?equipo
```

### Obtener el máximo:
```sparql
SELECT (MAX(?visitantes) AS ?maxVisitantes) {
    # consulta para visitantes
}
GROUP BY ?museo
```

## 11. ORDER BY - Ordenar Resultados

Se indica al final de la consulta: 

```sparql
ORDER BY DESC(?año)        # Descendente
ORDER BY ASC(?año)         # Ascendente
```



## 14. Consejos Prácticos

1. **Siempre usa DISTINCT** si sospechas que puede haber duplicados
2. **OPTIONAL evita consultas vacías y fallos** cuando no todas las entidades tienen todos los datos.
3. **Filtra el idioma** para obtener nombres legibles y evitar filas duplicadas con propiedades en varios idiomas
4. **Usa LIMIT** para no saturar el servidor mientras pruebas
5. **GROUP BY** es necesario cuando usas GROUP_CONCAT o funciones de agregación
6. **Termina cada tripleta con punto (.)** excepto la última del bloque
7. **Usa punto y coma (;)** para continuar agregando predicados al mismo sujeto
8. **Los comentarios** se escriben con #

## 15. Patrones Comunes

### Obtener entidades con sus nombres:
```sparql
?entidad wdt:P31 wd:QXXXXX ;
         rdfs:label ?nombre .
FILTER (lang(?nombre) = 'en')
```

### Navegar relaciones:
```sparql
?entidad wdt:P123 ?editorialEntity .
?editorialEntity rdfs:label ?editorial .
FILTER (lang(?editorial) = 'en')
```

### Fechas y años:
```sparql
?entidad wdt:P577 ?fecha .
BIND(YEAR(?fecha) AS ?año)
FILTER (?año >= 2000 && ?año <= 2024)
```

### Múltiples valores concatenados:
```sparql
SELECT ?entidad (GROUP_CONCAT(DISTINCT ?valor; separator=", ") AS ?valores)
WHERE {
    ?entidad wdt:PXXXX ?valorEntity .
    ?valorEntity rdfs:label ?valor .
    FILTER (lang(?valor) = 'en')
}
GROUP BY ?entidad
```

# **Práctica realizada por Pedro Antonio Martín Peláez y Álvaro Ray Martínez**

# Practica consultas SPARQL con WIKIDATA

Utilizar la página de consultas de WIKIDATA para ejecutar las siguientes consultas SPAQL y averiguar cómo conseguir las que se dejan como ejercicio.

In [1]:
# EJECUTA ESTA CELDA PARA ABRIR LA PÁGINA DE CONSULTAS DE WIKIDATA


# Utiliza esta dirección para abrir la página de consultas de DBPEDIA
import webbrowser
import urllib

main_url = "https://query.wikidata.org/"

# Función para abrir la URL con la consulta SPARQL
def open_query(query=""):
    url = main_url + "#" + urllib.parse.quote_plus(query).replace("+", "%20")
    webbrowser.open(url, new=0, autoraise=True)


open_query()


## Dominio 1: Películas y series

### Consulta 1.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener películas de Chistopher Nolan y año de estreno

In [2]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pelicula ?titulo ?año
WHERE {
    ?pelicula wdt:P31 wd:Q11424 ;         # instancia de: película
            wdt:P57 wd:Q25191 ;          # director: Christopher Nolan (Q25191)
            rdfs:label ?titulo ;
            wdt:P577 ?fecha .            # fecha de publicación
    
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?titulo) = 'en')
}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 1.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener series de televisión con su creador y año de inicio

In [3]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?titulo ?director ?año
WHERE {
    ?serie wdt:P31 wd:Q5398426 ; 
            rdfs:label ?titulo; # instancia de: serie de television
            wdt:P170 ?creator ; # creator
            wdt:P577 ?fecha .
    ?creator rdfs:label ?director .
    BIND(YEAR(?fecha) AS ?año)
    FILTER (lang(?titulo) = 'en') #para filtrar titulos y creador en un solo idioma
    FILTER (lang(?director) = 'en')
}
ORDER BY ASC(?titulo)
"""
open_query(query)

### Consulta 1.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Películas de Christopher Nolan con duración, presupuesto y recaudación


In [4]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?titulo ?duracion ?presupuesto ?recaudacion
WHERE {
    ?pelicula wdt:P31 wd:Q11424 ;         # instancia de: película
            wdt:P57 wd:Q25191 ;         # director: Christopher Nolan (Q25191)
            rdfs:label ?titulo .
        
    OPTIONAL { ?pelicula wdt:P2047 ?duracion }      # duración
    OPTIONAL { ?pelicula wdt:P2130 ?presupuesto }   # coste
    OPTIONAL { ?pelicula wdt:P2142 ?recaudacion }   # ingresos de taquilla

    FILTER (lang(?titulo) = 'en')
}
ORDER BY DESC(?recaudacion)
LIMIT 30
"""
open_query(query)

### Consulta 1.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Series de TV españolas con creador,  número de temporadas, episodios y cadena


In [5]:
query = """

PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?titulo ?creator ?num_t ?eps ?cadena_nombre
WHERE {
    ?serie wdt:P31 wd:Q5398426 ;         # instancia de: serie
            wdt:P170 ?director ;         # tenga director
            rdfs:label ?titulo ;
            wdt:P495 wd:Q29 ;            #sea española
            wdt:P2437 ?num_t ;
            wdt:P1113 ?eps ;
            wdt:P449 ?cadena .
    ?director rdfs:label ?creator .
    ?cadena rdfs:label ?cadena_nombre .
    FILTER (lang(?titulo) = 'es')
    FILTER (lang(?creator) = 'es')
    FILTER (lang(?cadena_nombre) = 'es')
}

"""
open_query(query)

## Dominio 2: Música

### Consulta 2.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener canciones de Michael Jackson

In [6]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?cancion ?titulo
WHERE {
    ?cancion wdt:P31 wd:Q7366 ;           # instancia de: canción
            wdt:P175 wd:Q2831 ;           # intérprete: Michael Jackson
            rdfs:label ?titulo .
    
    FILTER (lang(?titulo) = 'en')
}
LIMIT 50
"""
open_query(query)

### Consulta 2.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener álbumes de "The Beatles" con fecha de publicación


In [7]:
query = """

PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pelicula ?titulo ?fecha
WHERE {
    ?pelicula wdt:P31 wd:Q5398426 ;         # instancia de: serie de television
            wdt:P170 ?creator ;          # director: Christopher Nolan (Q25191)
            rdfs:label ?titulo ;
            wdt:P577 ?fecha .            # fecha de publicación
  
    FILTER (lang(?titulo) = 'en')
}
ORDER BY DESC(?año)

"""
open_query(query)

### Consulta 2.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Canciones con duración, género musical y compositor


In [8]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?cancion ?titulo ?fecha ?album
WHERE {
    ?cancion wdt:P31 wd:Q55850593 ;  
             wdt:P175 wd:Q2831; # instancia de: canción
             rdfs:label ?titulo .
    
    OPTIONAL {
        ?cancion wdt:P1433 ?albumEntidad.
        ?albumEntidad rdfs:label ?album . 
        ?albumEntidad wdt:P577 ?fecha .    
    }

    
    FILTER (lang(?titulo) = 'en')
    FILTER (lang(?album) = 'en')

}
ORDER BY DESC(?fecha)
LIMIT 50
"""
open_query(query)

### Consulta 2.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Álbumes con duración total, discográfica y productor

In [9]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?titulo ?duracion_total ?discografica ?productor
WHERE {
    ?album wdt:P31 wd:Q482994 ;           # instancia de: album
            rdfs:label ?titulo ;
            wdt:P2047 ?duracion_total;
            wdt:P483 ?disco;
            wdt:P162 ?p.
    ?p rdfs:label ?productor .
    ?disco rdfs:label ?discografica .
  
    FILTER (lang(?titulo) = 'en')
    FILTER (lang(?productor) = 'en')
    FILTER (lang(?discografica) = 'en')
}


"""
open_query(query)

## Dominio 3: Recetas y gastronomía


### Consulta 3.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener platos típicos de España

In [10]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?plato ?nombre
WHERE {
    ?plato wdt:P31 wd:Q746549 ; # instancia de: plato
           wdt:P495 wd:Q29;
           rdfs:label ?nombre.
    
    
    FILTER (lang(?nombre) = 'es')
}
LIMIT 50
"""
open_query(query)

### Consulta 3.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener bebidas con su país de origen

In [11]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?nombre ?pais_origen
WHERE {
    ?bebida wdt:P31 wd:Q40050 ; # instancia de: bebida
           wdt:P495 ?pais;
           rdfs:label ?nombre.
    ?pais rdfs:label ?pais_origen .
    FILTER (lang(?nombre) = 'en')
    FILTER (lang(?pais_origen) = 'en')
}
"""
open_query(query)

### Consulta 3.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Platos españoles con ingredientes principales y categoría culinaria


In [12]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?nombre ?ingrediente ?categoria
WHERE {
    ?plato wdt:P31 wd:Q746549 ;           # instancia de: plato
           wdt:P495 wd:Q29;
           rdfs:label ?nombre .
    
    OPTIONAL { ?plato wdt:P527 ?ingredienteEntity .  # consta de
               ?ingredienteEntity rdfs:label ?ingrediente .
               FILTER (lang(?ingrediente) = 'es')
    }
    
    OPTIONAL { ?plato wdt:P279 ?categoriaEntity .    # subclase de
               ?categoriaEntity rdfs:label ?categoria .
               FILTER (lang(?categoria) = 'es')
    }
    
    
    FILTER (lang(?nombre) = 'es')

  FILTER (BOUND(?ingrediente))
}
LIMIT 400
"""
open_query(query)

### Consulta 3.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Bebidas alcoholicas (de cualquier pais) con tipo, contenido alcohólico y fabricante


In [13]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?nombre ?tipoNombre ?contenido ?fabricanteNombre
WHERE {
  ?bebida wdt:P31 wd:Q154 ; 
          rdfs:label ?nombre ; # Instancia de: bebida alcohólica
          wdt:P279 ?tipo .
  
  ?tipo rdfs:label ?tipoNombre .
  
  OPTIONAL {
     ?bebida wdt:P2665 ?contenido .
  }
  
  OPTIONAL {
    ?bebida wdt:P176 ?fabricante .
    ?fabricante rdfs:label ?fabricanteNombre .
    FILTER (lang(?fabricanteNombre) = "en")
  }

  FILTER (lang(?nombre) = "en")
  FILTER (lang(?tipoNombre) = "en")
}
"""
open_query(query)

## Dominio 4: Productos tecnológicos


### Consulta 4.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener smartphones con su año de lanzamiento


In [14]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?telefono ?nombre ?año
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;         # instancia de: smartphone
              rdfs:label ?nombre ;
              wdt:P571 ?fecha .
        
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?nombre) = 'en')
}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 4.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener teléfonos con su fabricante y año de lanzamiento


In [15]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?nombre ?fabricante ?año
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;         # instancia de: smartphone
              rdfs:label ?nombre ;
              wdt:P176 ?f ;
              wdt:P571 ?fecha .
  
    ?f rdfs:label ?fabricante.
        
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?nombre) = 'en')
    FILTER (lang(?fabricante) = 'en')
}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 4.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Smartphones con fabricante año, cpu y storage


In [16]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?telefono ?nombre ?manufacturer ?año ?cpu ?storage
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;         # instancia de: smartphone
              rdfs:label ?nombre .
        
    OPTIONAL {
      ?telefono wdt:P577 ?fecha.
      BIND(YEAR(?fecha) AS ?año)
    }

    OPTIONAL {
       ?telefono wdt:P176 ?manufacturerEntity.
       ?manufacturerEntity rdfs:label ?manufacturer.
       FILTER (lang(?manufacturer) = 'en')
    }
  
    OPTIONAL {
       ?telefono wdt:P880 ?cpuEntity.
       ?cpuEntity rdfs:label ?cpu.
    } 
  
     OPTIONAL {
       ?telefono wdt:P2928 ?storage.
    } 

  
    FILTER (lang(?nombre) = 'en')

}
ORDER BY DESC(?año)
"""
open_query(query)

### Consulta 4.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Smartphones con fabricante año, cpu, storage y sistema operativo


In [17]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?nombre ?fabricante ?año ?cpu ?sistema
WHERE {
    ?telefono wdt:P31 wd:Q19723451 ;         # instancia de: smartphone
              rdfs:label ?nombre ;
              wdt:P176 ?f ;
              wdt:P571 ?fecha ;
              wdt:P880 ?c ;
              wdt:P306 ?s.
  
    ?f rdfs:label ?fabricante .
    ?c rdfs:label ?cpu .
    ?s rdfs:label ?sistema .
        
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?nombre) = 'en')
    FILTER (lang(?fabricante) = 'en')
    FILTER (lang(?cpu) = 'en')
    FILTER (lang(?sistema) = 'en')
}
"""
open_query(query)

## Dominio 5: Turismo y viajes


### Consulta 5.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener monumentos de España con su ciudad


In [18]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?monumento ?nombre ?ciudad
WHERE {
    ?monumento wdt:P31/wdt:P279*  wd:Q4989906 ;      # instancia de: monumento
               rdfs:label ?nombre ;
               wdt:P17 wd:Q29;
               wdt:P131 ?ciudadEntity .   # ubicado en
    
    ?ciudadEntity rdfs:label ?ciudad.
      
    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?ciudad) = 'es')
}
"""
open_query(query)

### Consulta 5.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener museos de historia de la Unión Europea con su ciudad y país


In [19]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?nombre ?pais ?ciudad
WHERE {
    ?m wdt:P31 wd:Q16735822 ;      # instancia de: museo historia
               rdfs:label ?nombre ;
               wdt:P17 ?p ;
               wdt:P131 ?ciudadEntity .   # ubicado en
    
    
    ?p rdfs:label ?pais;
       wdt:P361 wd:Q458 .
    
    ?ciudadEntity rdfs:label ?ciudad .
  
  
    FILTER (lang(?nombre) = 'en')
    FILTER (lang(?pais) = 'en')
    FILTER (lang(?ciudad) = 'en')
}
LIMIT 100

"""
open_query(query)

### Consulta 5.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Seleccioar los 10 museos españoles con mayor número de visitantes


In [20]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?museo ?nombre ?ciudad (MAX(?visitantes) AS ?maxVisitantes)
WHERE {
    ?museo wdt:P31/wdt:P279* wd:Q33506 ;      # instancia de: monumento
               rdfs:label ?nombre ;
               wdt:P17 wd:Q29;                # ubicado en España
               wdt:P131 ?ciudadEntity .       # ubicado en ciudad
    
    ?ciudadEntity rdfs:label ?ciudad.
    
    OPTIONAL {
         ?museo wdt:P1174 ?visitantes
    }
      
    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?ciudad) = 'es')
}
GROUP BY ?museo ?nombre ?ciudad
ORDER BY DESC(?maxVisitantes)
LIMIT 10
"""
open_query(query)

### Consulta 5.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Aeropuertos internacionales de España con código IATA, elevación y lugar. Si existen varios lugares se concatenan con GROUP_CONCAT. P.e.: (GROUP_CONCAT(DISTINCT ?lugar; separator=", ") AS ?lugares)


In [21]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?nombreAirport (GROUP_CONCAT(DISTINCT ?lugar; separator=", ") AS ?lugares)?iataCode ?paisName
WHERE {
  ?a wdt:P31 wd:Q644371;      # instancia de aeropuerto
     wdt:P17 wd:Q29;          # ubicado en España
     wdt:P238 ?iataCode;      # tiene código IATA
     rdfs:label ?nombreAirport;
     wdt:P131 ?l.             # ubicación administrativa (ciudad/provincia)

  ?l rdfs:label ?lugar.
  wd:Q29 rdfs:label ?paisName. # nombre de España

  FILTER (lang(?nombreAirport) = 'es')
  FILTER (lang(?lugar) = 'es')
  FILTER (lang(?paisName) = 'es')
}
GROUP BY ?nombreAirport ?iataCode ?paisName
LIMIT 100
"""
open_query(query)

## Dominio 6: Inmobiliario


### Consulta 6.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener edificios situados en las calles conectadas a la Calle Alcalá (Madrid)


In [22]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?calles ?edificio
WHERE {
    ?callesEntity wdt:P2789 wd:Q2424746.        # instancia de: edificio
    ?callesEntity rdfs:label ?calles.
    
    ?edificioEntity wdt:P669 ?callesEntity.
    ?edificioEntity rdfs:label ?edificio.
    FILTER (lang(?calles) = 'es')
    FILTER (lang(?edificio) = 'es')

}
"""
open_query(query)

### Consulta 6.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener edificios situados en las calles conectadas a la Calle Alcalá (Madrid) que estén declarados bien de interes cultural y fueron creados antes de 1900.

In [23]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?calles ?edificio ?año
WHERE {
    ?callesEntity wdt:P2789 wd:Q2424746.        # instancia de: edificio
    ?callesEntity rdfs:label ?calles.
    
    ?edificioEntity wdt:P669 ?callesEntity;
                    wdt:P571 ?fecha ;
                    rdfs:label ?edificio ;
                    wdt:P1435 ?interes.
  
    BIND(YEAR(?fecha) AS ?año)
  
    FILTER (lang(?calles) = 'es')
    FILTER (lang(?edificio) = 'es')
    FILTER (?año < 1900)
}
"""
open_query(query)

### Consulta 6.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Edificios de Nueva York con altura mayor de 50 metros, número de plantas y arquitecto

In [24]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?edificio ?nombre ?altura ?plantas ?arquitecto
WHERE {
    ?edificio wdt:P31 wd:Q41176 ;         # instancia de: edificio
              rdfs:label ?nombre ;
              wdt:P131 wd:Q60 .
    
    
    OPTIONAL { ?edificio wdt:P2048 ?altura }  # altura en metros
    
    OPTIONAL { ?edificio wdt:P1101 ?plantas }  # número de plantas
    
    OPTIONAL { ?edificio wdt:P84 ?arquitectoEntity .
               ?arquitectoEntity rdfs:label ?arquitecto .
               FILTER (lang(?arquitecto) = 'en')
    }
    
    FILTER (lang(?nombre) = 'en')
    FILTER (BOUND(?altura) && ?altura > 50)
}
ORDER BY DESC(?altura)
LIMIT 30
"""
open_query(query)


### Consulta 6.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Rascacielos con altura mayor que 200, coste de construcción, número de plantas, y usos (concatenados con GROUP_CONCAT)

In [25]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?rascacielo ?altura ?coste ?plantas (GROUP_CONCAT(DISTINCT ?usoLabel; separator=", ") AS ?usos)
WHERE {
    ?bldg wdt:P31 wd:Q11303 ;        # instancia de: rascacielos
          wdt:P2048 ?altura ;        # altura
          wdt:P1101 ?plantas ;       # número de plantas
          wdt:P366 ?uso ;
          rdfs:label ?rascacielo;
          wdt:P2130 ?coste .

    ?uso rdfs:label ?usoLabel .
  
    FILTER(LANG(?usoLabel) = "en")
    FILTER(LANG(?rascacielo) = "en")
    FILTER(?altura > 200)
}
GROUP BY ?rascacielo ?altura ?coste ?plantas


"""
open_query(query)

## Dominio 7: Libros y literatura


### Consulta 7.1 - Nivel Básico (Ejemplo)
**Objetivo**: Series de manga creadas por Akira Toriyama, junto con número de partes que las componen y publisher.

In [26]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?libro ?titulo ?partes ?publisher
WHERE {
    ?libro wdt:P50 wd:Q208582 ;          # instancia de: obra literaria
           wdt:P31 wd:Q21198342;
           rdfs:label ?titulo .
  
    OPTIONAL {?libro wdt:P2635 ?partes.} 
    OPTIONAL {?libro wdt:P123 ?publisherEntity. 
             ?publisherEntity rdfs:label ?publisher.
             FILTER (lang(?publisher) = 'en')
    }
   
    FILTER (lang(?titulo) = 'en')
}
"""
open_query(query)

### Consulta 7.2 - Nivel Básico (Ejercicio)
**Objetivo**: Comics Españoles publicados antes de 1990, con título, año y creadores (concatenados si hay varios)

In [27]:
query = """

SELECT DISTINCT ?titulo ?año (GROUP_CONCAT(DISTINCT ?creatorLabel; separator=", ") AS ?creadores)
WHERE {
    ?comic wdt:P31/wdt:P279* wd:Q1004 ;         # que sea comic instacia o subclase
           wdt:P495 wd:Q29;                     # de españa
           rdfs:label ?titulo;
           wdt:P580 ?date;
           wdt:P170 ?creator.
  
       ?creator rdfs:label ?creatorLabel
  
       BIND(YEAR(?date) AS ?año)
       FILTER (lang(?titulo) = 'es')
       FILTER (lang(?creatorLabel) = 'es')
       FILTER(?año < 1990)
}

GROUP BY ?titulo ?año

"""
open_query(query)

### Consulta 7.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Libros publicados por Perez-Reverte con editorial y género literario


In [28]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?titulo ?editorial ?genero
WHERE {
    ?libro wdt:P31/wdt:P279* wd:Q7725634 ;          # instancia de: obra literaria
           rdfs:label ?titulo ;
           wdt:P50 wd:Q155106 .
    
    
    OPTIONAL { ?libro wdt:P123 ?editorialEntity .
               ?editorialEntity rdfs:label ?editorial .
               FILTER (lang(?editorial) = 'en')
    }
    OPTIONAL { ?libro wdt:P136 ?generoEntity .
               ?generoEntity rdfs:label ?genero .
               FILTER (lang(?genero) = 'en')
    }
    
    FILTER (lang(?titulo) = 'es')
}
"""
open_query(query)

### Consulta 7.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Libros con ubicación narrativa en Madrid, incluyendo autor, año, editorial y género. Ordenados por fecha de publicación.



In [29]:
query = """

SELECT DISTINCT ?titulo ?año ?autorLabel ?editorialLabel ?generoLabel
WHERE {
    ?libro wdt:P31 wd:Q47461344 ;        # que sea libro
           wdt:P840 wd:Q2807 ;      # ubicación narrativa: Madrid
           rdfs:label ?titulo ;
           wdt:P577 ?fecha.
           
    OPTIONAL{?libro wdt:P136 ?genero. ?genero rdfs:label ?generoLabel . FILTER (lang(?generoLabel) = 'es')}
    OPTIONAL{?libro wdt:P123 ?editorial. ?editorial rdfs:label ?editorialLabel. FILTER (lang(?editorialLabel) = 'es')}
    OPTIONAL{?libro wdt:P50 ?autor. ?autor rdfs:label ?autorLabel . FILTER (lang(?autorLabel) = 'es')}
    
    

    BIND(YEAR(?fecha) AS ?año)
    FILTER (lang(?titulo) = 'es')
    
    
    
}
ORDER BY ?año
Limit 100

"""
open_query(query)

## Dominio 8: Salud y síntomas médicos


### Consulta 8.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener enfermedades cuya causa sea fumar, con sus síntomas principales agrupadas

In [30]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?enfermedad ?nombre (GROUP_CONCAT(DISTINCT ?sintoma; separator=", ") AS ?sintomas) 
WHERE {
    ?enfermedad wdt:P31/wdt:P279* wd:Q12136 ;       # instancia de: enfermedad
                wdt:P828 wd:Q662860;
                rdfs:label ?nombre .
    
    OPTIONAL { ?enfermedad wdt:P780 ?sintomaEntity .
               ?sintomaEntity rdfs:label ?sintoma .
               FILTER (lang(?sintoma) = 'en')
    }
    
    FILTER (lang(?nombre) = 'en')
}
GROUP BY ?enfermedad ?nombre
LIMIT 50
"""
open_query(query)

### Consulta 8.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener medicamentos (Límite 50) con sus usos terapéuticos


In [31]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?medicamento ?uso_terapeutico
WHERE {
    ?m wdt:P31 wd:Q12140;          # instancia de: medicamento
           rdfs:label ?medicamento ;
           wdt:P2175 ?uso.
  
    ?uso rdfs:label ?uso_terapeutico.
   
    FILTER (lang(?medicamento) = 'en')
    FILTER (lang(?uso_terapeutico) = 'en')
}
LIMIT 50
"""
open_query(query)

### Consulta 8.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Enfermedades con causas, tratamientos y especialidad médica


In [32]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?nombre ?causa ?especialidad (GROUP_CONCAT(DISTINCT ?tratamiento; separator=", ") AS ?tratamientos) 
WHERE {
    ?enfermedad wdt:P31/wdt:P279* wd:Q12136 ;       # instancia de: enfermedad
                rdfs:label ?nombre .
    
    OPTIONAL { ?enfermedad wdt:P828 ?causaEntity .
               ?causaEntity rdfs:label ?causa .
               FILTER (lang(?causa) = 'en')
    }
    
    OPTIONAL { ?enfermedad wdt:P2176 ?tratamientoEntity .
               ?tratamientoEntity rdfs:label ?tratamiento .
               FILTER (lang(?tratamiento) = 'en')
    }
    
    OPTIONAL { ?enfermedad wdt:P1995 ?especialidadEntity .
               ?especialidadEntity rdfs:label ?especialidad .
               FILTER (lang(?especialidad) = 'en')
    }
    
    FILTER (lang(?nombre) = 'en')
    FILTER (BOUND(?especialidad))
}
GROUP BY ?nombre ?causa ?especialidad
LIMIT 30
"""
open_query(query)

### Consulta 8.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Medicamentos con mecanismo de acción, vía de administración, fabricante y efectos


In [33]:
query = """
SELECT DISTINCT ?medicamentoLabel ?mecanismoLabel ?viaLabel ?fabricanteLabel ?efectoLabel
WHERE {
    # Esto es obligatorio: que sea un medicamento y tenga etiqueta en inglés.
    ?medicamento wdt:P31 wd:Q12140 ;
                 rdfs:label ?medicamentoLabel .
    OPTIONAL{
      ?medicamento wdt:P636 ?via .
      ?via rdfs:label ?viaLabel .
      FILTER(lang(?viaLabel)="en")
    }


    OPTIONAL {
        ?medicamento wdt:P3771 ?mecanismo .
        ?mecanismo rdfs:label ?mecanismoLabel .
        FILTER(lang(?mecanismoLabel)="en")
    }
    OPTIONAL {
        ?medicamento wdt:P176 ?fabricante .
        ?fabricante rdfs:label ?fabricanteLabel .
        FILTER(lang(?fabricanteLabel)="en")
    }
    OPTIONAL {
        ?medicamento wdt:P1542 ?efecto .
        ?efecto rdfs:label ?efectoLabel .
        FILTER(lang(?efectoLabel)="en")
    }

    FILTER(lang(?medicamentoLabel)="en")

}
"""
open_query(query)

## Dominio 9: Deportes (Fútbol)

### Consulta 9.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener futbolistas españoles con su posición y equipo actual


In [34]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?jugador ?nombre ?posicion ?equipo
WHERE {
    ?jugador wdt:P31 wd:Q5 ;              # instancia de: humano
             wdt:P106 wd:Q937857 ;         # ocupación: futbolista
             rdfs:label ?nombre ;
             wdt:P413 ?posicionEntity ;
             wdt:P54 ?equipoEntity .

    ?jugador wdt:P27 wd:Q29.
    ?posicionEntity rdfs:label ?posicion .
    ?equipoEntity rdfs:label ?equipo .
    
    FILTER (lang(?nombre) = 'es')
    FILTER (lang(?posicion) = 'es')
    FILTER (lang(?equipo) = 'es')
}
LIMIT 50
"""
open_query(query)

### Consulta 9.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener equipos españoles de la liga


In [35]:
query = """
SELECT DISTINCT ?equipo
WHERE {
  ?e wdt:P31 wd:Q476028 ;       # Instancia de club de fútbol
          wdt:P17 wd:Q29 ;          # País = España
          wdt:P118 wd:Q324867 ;
          rdfs:label ?equipo.

    FILTER (lang(?equipo) = 'es')
}
"""
open_query(query)

### Consulta 9.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Futbolistas del Real Madrid con estadísticas físicas (altura, peso, fecha de nacimiento)

In [36]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?nombre ?altura ?peso ?fechaNacimiento ?pais
WHERE {
    ?jugador wdt:P31 wd:Q5 ;              # instancia de: humano
             wdt:P106 wd:Q937857 ;         # ocupación: futbolista
             wdt:P54 wd:Q8682;
             rdfs:label ?nombre .
    
    OPTIONAL { ?jugador wdt:P2048 ?altura }
    OPTIONAL { ?jugador wdt:P2067 ?peso }
    OPTIONAL { ?jugador wdt:P569 ?fechaNacimiento }
    OPTIONAL { ?jugador wdt:P27 ?paisEntity .
               ?paisEntity rdfs:label ?pais .
               FILTER (lang(?pais) = 'es')
    }
    
    FILTER (lang(?nombre) = 'es')
    FILTER (BOUND(?altura) && ?altura > 1.70 && ?altura < 2.00)
}
ORDER BY DESC(?altura)
LIMIT 40
"""
open_query(query)

### Consulta 9.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Equipos de fútbol de La Liga con estadio, año de fundación, entrenador y número de títulos

In [37]:
query = """
SELECT ?equipo ?estadio ?fundacion ?entrenador (COUNT(?titulo) AS ?numTitulos)
WHERE {
  ?club wdt:P31 wd:Q476028 ;     # Club de fútbol
        wdt:P17 wd:Q29 ;         # España
        wdt:P118 wd:Q324867 ;    # La Liga
        rdfs:label ?equipo ;
        wdt:P115 ?estadioEnt ;
        wdt:P571 ?fundacion ;
        wdt:P286 ?entrenadorEnt .

  ?titulo wdt:P1346 ?club .

  ?estadioEnt rdfs:label ?estadio .
  ?entrenadorEnt rdfs:label ?entrenador .

  FILTER(LANG(?equipo) = "es")
  FILTER(LANG(?estadio) = "es")
  FILTER(LANG(?entrenador) = "es")
}
GROUP BY ?equipo ?estadio ?fundacion ?entrenador
"""
open_query(query)

## Dominio 10: Arte y museos


### Consulta 10.1 - Nivel Básico (Ejemplo)
**Objetivo**: Obtener pinturas españolas con su autor y año de creación


In [38]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pintura ?titulo ?artista ?año
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;        # instancia de: pintura
             wdt:P17 wd:Q29;
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity ;
             wdt:P571 ?fecha .
    
    ?artistaEntity rdfs:label ?artista .
    
    BIND(YEAR(?fecha) AS ?año)
    
    FILTER (lang(?titulo) = 'en')
    FILTER (lang(?artista) = 'en')
}
ORDER BY DESC(?año)
LIMIT 50
"""
open_query(query)

### Consulta 10.2 - Nivel Básico (Ejercicio)
**Objetivo**: Obtener pinturas del Museo del Prado anteriores a 1700 con su autor y año de creación

In [39]:
query = """

SELECT ?titulo ?artista ?año
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;        # instancia de: pintura
             wdt:P276 wd:Q160112;          # del mueso del prado
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity ;
             wdt:P571 ?fecha .
    
    ?artistaEntity rdfs:label ?artista .
    
    BIND(YEAR(?fecha) AS ?año)
    FILTER(?año < 1700)
    
    FILTER (lang(?titulo) = 'es')
    FILTER (lang(?artista) = 'es')
}
ORDER BY DESC(?año)
LIMIT 50


"""
open_query(query)

### Consulta 10.3 - Nivel Avanzado (Ejemplo)
**Objetivo**: Pinturas con movimiento artístico, ubicación actual y material


In [40]:
query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?titulo ?artista ?movimiento ?museo ?material
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;        # instancia de: pintura
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity .
    
    ?artistaEntity rdfs:label ?artista .
    
    OPTIONAL { ?pintura wdt:P135 ?movimientoEntity .
               ?movimientoEntity rdfs:label ?movimiento .
               FILTER (lang(?movimiento) = 'en')
    }
    
    OPTIONAL { ?pintura wdt:P276 ?museoEntity .
               ?museoEntity rdfs:label ?museo .
               FILTER (lang(?museo) = 'en')
    }
    
    OPTIONAL { ?pintura wdt:P186 ?materialEntity .
               ?materialEntity rdfs:label ?material .
               FILTER (lang(?material) = 'en')
    }
    
    FILTER (lang(?titulo) = 'en')
    FILTER (lang(?artista) = 'en')
    FILTER (BOUND(?movimiento))
}
LIMIT 30
"""
open_query(query)

### Consulta 10.4 - Nivel Avanzado (Ejercicio)
**Objetivo**: Pinturas del Museo del Prado hechas por artistas españoles antes del año 1700, incluyendo movimiento y materiales (agrupados)


In [41]:
query = """
SELECT ?titulo ?artista ?año (GROUP_CONCAT(DISTINCT ?movimiento; separator=", ") AS ?movimientos)
                               (GROUP_CONCAT(DISTINCT ?material; separator=", ") AS ?materiales)
WHERE {
    ?pintura wdt:P31 wd:Q3305213 ;       # Instancia de: pintura
             wdt:P195 wd:Q160112 ;      # Museo del Prado
             rdfs:label ?titulo ;
             wdt:P170 ?artistaEntity ;
             wdt:P571 ?fecha ;
             wdt:P135 ?movimientoEntity ;
             wdt:P186 ?materialEntity .

    ?artistaEntity rdfs:label ?artista ;
                   wdt:P27 wd:Q29 .      # Nacionalidad española

    ?movimientoEntity rdfs:label ?movimiento .
    ?materialEntity rdfs:label ?material .

    BIND(YEAR(?fecha) AS ?año)

    FILTER(?año < 1700)
    FILTER(LANG(?titulo) = "es")
    FILTER(LANG(?artista) = "es")
    FILTER(LANG(?movimiento)="es")
    FILTER(LANG(?material)="es")
}
GROUP BY ?titulo ?artista ?año
LIMIT 50
"""
open_query(query)